In [ ]:
# Load results from all seeds and create plots.
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Configs:
BASE_CACHE = os.environ.get("BASE_CACHE", os.path.join(os.getcwd()))
SEEDS = [3, 17, 24]
# Load results:
all_seed_results = []
print("Loading seed results")
for seed in SEEDS:
    seed_dir = os.path.join(BASE_CACHE, "runs", f"seed_{seed}")
    result_path = os.path.join(seed_dir, f"seed_{seed}_results.json")
    if os.path.exists(result_path):
        with open(result_path, "r") as f:
            seed_result = json.load(f)
        all_seed_results.append(seed_result)
        print(f"Loaded seed {seed}")
    else:
        print(f"Missing seed {seed}: {result_path}")
if len(all_seed_results) == 0:
    print("ERROR: No seed results found!")
    print("Make sure you've run single_seed_run.py for all seeds first.")
    exit(1)

print(f"\nLoaded {len(all_seed_results)} seeds\n")

# Helper Function:
def extract_metric(seed_results, variant, metric):
    return np.array([s[variant][metric] for s in seed_results], dtype=np.float64)

def centered_moving_average(x, window=3):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    if x.size < window:
        return x
    k = np.ones(window, dtype=np.float64) / window
    left = window // 2
    right = window - 1 - left
    padded = np.pad(x, (left, right), mode="edge")
    return np.convolve(padded, k, mode="valid")

def forward_fill_finite(x, default=1e-12):
    x = np.asarray(x, dtype=np.float64).copy()
    if x.size == 0:
        return x
    if not np.isfinite(x[0]) or x[0] <= 0:
        x[0] = default
    for i in range(1, x.size):
        if not np.isfinite(x[i]) or x[i] <= 0:
            x[i] = x[i - 1]
    return x

OUTPUT_DIR = os.path.join(BASE_CACHE, "runs", "aggregated_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("AGGREGATE ANALYSIS ACROSS SEEDS")
print("\nSummary (mean ± std)")
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    final_a = extract_metric(all_seed_results, variant, "final_a")
    final_b = extract_metric(all_seed_results, variant, "final_b")
    forgetting_a = extract_metric(all_seed_results, variant, "forgetting_a")
    gain_b = extract_metric(all_seed_results, variant, "gain_b")

    print(
        f"{variant:20s} | "
        f"final_A={100 * final_a.mean():.2f}%±{100 * final_a.std():.2f}% | "
        f"final_B={100 * final_b.mean():.2f}%±{100 * final_b.std():.2f}% | "
        f"forgetting_A={100 * forgetting_a.mean():+.2f}%±{100 * forgetting_a.std():.2f}% | "
        f"gain_B={100 * gain_b.mean():+.2f}%±{100 * gain_b.std():.2f}%"
    )
    
# Plots
# Plot 1: Task A WER Curve
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    a_curves = np.stack([s[variant]["a_hist"] for s in all_seed_results], axis=0)
    eval_steps = np.array(all_seed_results[0][variant]["eval_steps"], dtype=np.int64)
    mean = a_curves.mean(axis=0)
    std = a_curves.std(axis=0, ddof=0)
    raw = 100 * mean
    plt.plot(eval_steps, raw, label=f"{variant} raw", marker="o", linestyle="--", alpha=0.35, linewidth=1.5)
    plt.fill_between(eval_steps, 100 * (mean - std), 100 * (mean + std), alpha=0.12)
    sm = centered_moving_average(raw, window=3)
    plt.plot(eval_steps, sm, label=f"{variant} smoothed", linewidth=2.5)

plt.axhline(100 * all_seed_results[0]["a_before_b"], color="gray", linestyle="--", alpha=0.5, linewidth=2, label="A pre-B")
plt.xlabel("Train step", fontsize=12)
plt.ylabel("WER (%)", fontsize=12)
plt.title(f"Task A WER Evolution - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "01_task_a_wer.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

# Plot 2: Task B WER Curve
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    b_curves = np.stack([s[variant]["b_hist"] for s in all_seed_results], axis=0)
    eval_steps = np.array(all_seed_results[0][variant]["eval_steps"], dtype=np.int64)
    mean = b_curves.mean(axis=0)
    std = b_curves.std(axis=0, ddof=0)
    raw = 100 * mean
    plt.plot(eval_steps, raw, label=f"{variant} raw", marker="o", linestyle="--", alpha=0.35, linewidth=1.5)
    plt.fill_between(eval_steps, 100 * (mean - std), 100 * (mean + std), alpha=0.12)
    sm = centered_moving_average(raw, window=3)
    plt.plot(eval_steps, sm, label=f"{variant} smoothed", linewidth=2.5)

plt.axhline(100 * all_seed_results[0]["b_before_b"], color="gray", linestyle="--", alpha=0.5, linewidth=2, label="B pre-B")
plt.xlabel("Train step", fontsize=12)
plt.ylabel("WER (%)", fontsize=12)
plt.title(f"Task B WER Evolution - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "02_task_b_wer.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

# Plot 3: EWC Penalty per Step
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    penalty_curves = np.stack([s[variant]["ewc_hist"] for s in all_seed_results], axis=0)
    train_steps = all_seed_results[0][variant]["steps"]
    mean = penalty_curves.mean(axis=0)
    std = penalty_curves.std(axis=0, ddof=0)
    plt.plot(train_steps, mean, label=f"{variant}", linewidth=2.5)
    plt.fill_between(train_steps, mean - std, mean + std, alpha=0.2)
plt.xlabel("Train step", fontsize=12)
plt.ylabel("EWC Penalty Loss", fontsize=12)
plt.title(f"EWC Penalty per Step - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
penalty_plot_path = os.path.join(OUTPUT_DIR, "03_ewc_penalty.png")
plt.savefig(penalty_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {penalty_plot_path}")


# Plot 4: Total Loss
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    total_curves = np.stack([s[variant]["total_hist"] for s in all_seed_results], axis=0)
    train_steps = all_seed_results[0][variant]["steps"]
    mean = total_curves.mean(axis=0)
    std = total_curves.std(axis=0, ddof=0)
    plt.plot(train_steps, mean, label=f"{variant}", linewidth=2.5)
    plt.fill_between(train_steps, mean - std, mean + std, alpha=0.2)
plt.xlabel("Train step", fontsize=12)
plt.ylabel("Loss (log scale)", fontsize=12)
plt.title(f"Total Loss (ASR + Penalty) - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.yscale("log")
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
total_loss_plot_path = os.path.join(OUTPUT_DIR, "11_total_loss.png")
plt.savefig(total_loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {total_loss_plot_path}")

# Plot 5: ASR Loss Only
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    asr_curves = np.stack([s[variant]["asr_hist"] for s in all_seed_results], axis=0)
    train_steps = all_seed_results[0][variant]["steps"]
    mean = asr_curves.mean(axis=0)
    std = asr_curves.std(axis=0, ddof=0)
    plt.plot(train_steps, mean, label=f"{variant}", linewidth=2.5)
    plt.fill_between(train_steps, mean - std, mean + std, alpha=0.2)
plt.xlabel("Train step", fontsize=12)
plt.ylabel("Loss (log scale)", fontsize=12)
plt.title(f"ASR Task Loss Only - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.yscale("log")
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
asr_loss_plot_path = os.path.join(OUTPUT_DIR, "12_asr_loss.png")
plt.savefig(asr_loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {asr_loss_plot_path}")

# Plot 6: EWC Loss Only
plt.figure(figsize=(10, 6))
for variant in ["baseline", "ewc", "mini_kfac_ewc"]:
    ewc_curves = np.stack([s[variant]["ewc_hist"] for s in all_seed_results], axis=0)
    train_steps = all_seed_results[0][variant]["steps"]
    mean = ewc_curves.mean(axis=0)
    std = ewc_curves.std(axis=0, ddof=0)
    plt.plot(train_steps, mean, label=f"{variant}", linewidth=2.5)
    plt.fill_between(train_steps, mean - std, mean + std, alpha=0.2)
plt.xlabel("Train step", fontsize=12)
plt.ylabel("Loss (log scale)", fontsize=12)
plt.title(f"EWC Regularization Loss Only - {len(all_seed_results)} seeds", fontsize=14, fontweight='bold')
plt.yscale("log")
plt.legend(fontsize=10, loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
ewc_loss_plot_path = os.path.join(OUTPUT_DIR, "13_ewc_loss.png")
plt.savefig(ewc_loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {ewc_loss_plot_path}")


# FINAL INTERPRETATION
print("\nFinal Interpretation")
forg_a_baseline = extract_metric(all_seed_results, "baseline", "forgetting_a").mean()
forg_a_ewc = extract_metric(all_seed_results, "ewc", "forgetting_a").mean()
forg_a_mini_kfac = extract_metric(all_seed_results, "mini_kfac_ewc", "forgetting_a").mean()
print(f"\nAbsolute Performance:")
print(f"  baseline: forgetting_A={100 * forg_a_baseline:+.2f}%")
print(f"  ewc: forgetting_A={100 * forg_a_ewc:+.2f}%")
print(f"  miniKFAC_EWC: forgetting_A={100 * forg_a_mini_kfac:+.2f}%")

delta_forg_a_ewc = forg_a_ewc - forg_a_baseline
delta_forg_a_mini_kfac = forg_a_mini_kfac - forg_a_baseline
print(f"\nImprovement vs Baseline:")
print(f"  ewc: forgetting_A={delta_forg_a_ewc*100:+.2f}%")
print(f"  miniKFAC_EWC: forgetting_A={delta_forg_a_mini_kfac*100:+.2f}%")
delta_forg_a = forg_a_mini_kfac - forg_a_ewc
print(f"\nminiKFAC_EWC vs EWC:")
print(f"  delta_forgetting_A: {100 * delta_forg_a:+.2f}%")

TIE_EPS = 0.003
if abs(delta_forg_a) <= TIE_EPS:
    winner = "tie"
elif delta_forg_a <= TIE_EPS:
    winner = "miniKFAC_EWC"
elif delta_forg_a >= -TIE_EPS:
    winner = "ewc"
else:
    winner = "tradeoff"
print(f"  winner: {winner}")

# Save summary
summary = {
    "num_seeds": len(all_seed_results),
    "seeds": SEEDS,
    "summary": {
        "baseline": {
            "forgetting_A": float(forg_a_baseline),
        },
        "ewc": {
            "forgetting_A": float(forg_a_ewc),
        },
        "miniKFAC_EWC": {
            "forgetting_A": float(forg_a_mini_kfac),
        },
    },
    "comparison": {
        "ewc_vs_baseline": {
            "delta_forgetting_A": float(delta_forg_a_ewc),
        },
        "miniKFAC_vs_baseline": {
            "delta_forgetting_A": float(delta_forg_a_mini_kfac),
        },
        "miniKFAC_vs_ewc": {
            "delta_forgetting_A": float(delta_forg_a),
            "winner": winner,
        },
    },
}

summary_path = os.path.join(OUTPUT_DIR, "summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nAll plots saved to: {OUTPUT_DIR}")
print(f"Summary saved to: {summary_path}")